# 02 — Split (train / val / test)

Definišemo i zamrzavamo podelu skupa na nivou cele slike:

- **TEST** = 70 celih slika nasumično odabranih kao test set u originalnom radu. Odabrano radi uporedivosti naših rezultata sa.
- Preostalih 213 slika -> **TRAIN/VAL** split 80/20
- fiksni `seed`, stratifikovano tako da udeo praznih maski ostane proporcionalan u train i val skupu.
- Ciljna veličina: val ≈ 20% od 213 (~43 slike), train ~170 (≈ 60/15/25 od ukupno 283).

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.split import make_splits, save_splits, _is_empty

## 1. Lista test slika

Ovih 70 imena je originalna test podela preuzeta iz repoa originalnog rada.

In [2]:
test_img_names = ['Mar21bS1C2R2_VLPAGl_200x_y.png',
                  'Mar20bS1C4R3_DMl_200x_y.png',
                  'Mar19bS1C5R3_DMr_200x_y.png',
                  'Mar27bS1C2R1_LHl_200x_y.png',
                  'Mar19bS1C4R1_VLPAGr_200x_y.png',
                  'Mar19bS1C3R2_VLPAGl_200x_y.png',
                  'Mar26bS1C4R3_DMl_200x_y.png',
                  'Mar21bS1C1R3_VLPAGr_200x_y.png',
                  'Mar20bS1C3R2_VLPAGl_200x_y.png',
                  'Mar22bS1C4R1_LHl_200x_y.png',
                  'Mar24bS1C3R2_LHr_200x_y.png',
                  'Mar20bS1C4R3_DMr_200x_y.png',
                  'Mar20bS1C2R3_VLPAGr_200x_y.png',
                  'Mar20bS1C2R2_VLPAGl_200x_y.png',
                  'Mar26bS1C4R2_LHl_200x_y.png',
                  'Mar19bS1C4R3_DMr_200x_y.png',
                  'Mar24bS1C2R2_LHl_200x_y.png',
                  'Mar20bS2C1R1_LHl_200x_y.png',
                  'Mar19bS1C1R2_VLPAGr_200x_y.png',
                  'Mar24bS2C4R3_DMr_200x_y.png',
                  'Mar23bS1C6R1_DMr_200x_y.png',
                  'Mar19bS1C3R2_VLPAGr_200x_y.png',
                  'Mar24bS1C1R1_LHl_200x_y.png',
                  'Mar21bS2C1R2_LHl_200x_y.png',
                  'Mar21bS1C2R3_VLPAGr_200x_y.png',
                  'Mar19bS1C2R3_VLPAGr_200x_y.png',
                  'Mar20bS2C1R3_DMl_200x_y.png',
                  'Mar26bS2C2R2_DMr_200x_y.png',
                  'Mar21bS1C2R1_VLPAGl_200x_y.png',
                  'Mar22bS1C4R2_LHr_200x_y.png',
                  'Mar22bS1C3R2_DMr_200x_y.png',
                  'Mar24bS2C2R3_VLPAGl_200x_y.png',
                  'Mar24bS1C2R1_DMl_200x_y.png',
                  'Mar20bS1C4R1_DMr_200x_y.png',
                  'Mar19bS1C5R2_DMr_200x_y.png',
                  'Mar20bS2C2R3_LHr_200x_y.png',
                  'Mar21bS2C2R2_LHr_200x_y.png',
                  'Mar20bS1C1R3_VLPAGr_200x_y.png',
                  'Mar22bS2C1R1_LHr_200x_y.png',
                  'Mar20bS1C2R1_VLPAGl_200x_y.png',
                  'Mar19bS1C5R3_LHl_200x_y.png',
                  'Mar26bS2C2R2_LHr_200x_y.png',
                  'Mar19bS1C5R2_DMl_200x_y.png',
                  'Mar26bS2C2R1_DMl_200x_y.png',
                  'Mar22bS1C2R1_VLPAGl_200x_y.png',
                  'Mar20bS2C2R3_LHl_200x_y.png',
                  'Mar19bS1C1R3_VLPAGl_200x_y.png',
                  'Mar27bS1C3R1_LHr_200x_y.png',
                  'Mar24bS1C3R1_LHr_200x_y.png',
                  'Mar24bS1C1R2_DMr_200x_y.png',
                  'Mar19bS1C4R2_LHr_200x_y.png',
                  'Mar21bS2C1R1_LHr_200x_y.png',
                  'Mar20bS1C4R1_LHl_200x_y.png',
                  'Mar26bS2C1R1_DMr_200x_y.png',
                  'Mar19bS1C5R2_LHl_200x_y.png',
                  'Mar21bS2C2R3_LHl_200x_y.png',
                  'Mar19bS1C4R3_LHr_200x_y.png',
                  'Mar23bS2C1R1_LHl_200x_y.png',
                  'Mar23bS1C1R4_VLPAGl_200x_y.png',
                  '39_y.png',
                  'Mar31bS2C1R2_VLPAGr_200x_y.png',
                  'Mar32bS2C2R2_DMl_200x_y.png',
                  'Mar33bS2C1R1_DMl_200x_y.png',
                  'MAR38S1C3R1_LHR_20_o.png',
                  'Mar42S2C4R2_VLPAGr_200x_o.png',
                  'Mar31bS2C3R4_DMr_200x_y.png',
                  'Mar33bS1C4R2_DMl_200x_y.png',
                  'Mar36bS1C6R2_DMr_200x_y.png',
                  'Mar42S2C2R2_DMr_200x_o.png',
                  'MAR55S1C5R3_DMR_20_o.png']

print("broj test imena:", len(test_img_names))

broj test imena: 70


## 2. Kreiranje i čuvanje splita

In [3]:
splits = make_splits(test_img_names)
save_splits(splits)
print("sačuvano u:", config.SPLITS_DIR)
for name, ids in splits.items():
    print(f"  {name}: {len(ids)} slika")

sačuvano u: /Users/ana_velimirovic/Desktop/Ana-priv/ML/projekat/Fluorescent-neuronal-cell-counting/data/splits
  train: 170 slika
  val: 43 slika
  test: 70 slika


## 3. Verifikacija

Sve provere dole moraju proći (nema curenja podataka, split je smislen).

In [4]:
train, val, test = splits["train"], splits["val"], splits["test"]

print("len(test) ==", len(test), "-> očekivano 70:", len(test) == 70)
print("len(train)+len(val)+len(test) ==", len(train)+len(val)+len(test), "-> očekivano 283:",
      len(train)+len(val)+len(test) == 283)

print("\ntrain ∩ test:", set(train) & set(test))
print("val ∩ test  :", set(val) & set(test))
print("train ∩ val :", set(train) & set(val))

all_ids = set(p.name for p in config.IMG_DIR.glob("*.png"))
print("\nsvih 70 test imena postoji u IMG_DIR:", set(test) <= all_ids)

len(test) == 70 -> očekivano 70: True
len(train)+len(val)+len(test) == 283 -> očekivano 283: True

train ∩ test: set()
val ∩ test  : set()
train ∩ val : set()

svih 70 test imena postoji u IMG_DIR: True


In [ ]:
# udeo praznih maski po skupu -> treba da bude ~isti svuda
rows = []
for name, ids in splits.items():
    empty_frac = sum(_is_empty(i) for i in ids) / len(ids)
    rows.append({"skup": name, "n_slika": len(ids), "udeo_praznih": round(empty_frac, 3)})

tabela = pd.DataFrame(rows).set_index("skup")
tabela

,n_slika,udeo_praznih
skup,,
train,170,0.176
val,43,0.186
test,70,0.171


## 4. Zaključak

test = originalnih 70 slika iz referentnog repoa, radi direktne uporedivosti naših rezultata sa njihovim rezultatima.
<br>
val =nizdvojen stratifikovano po "prazna maska" iz preostalih 213 slika, tako da udeo praznih ostane proporcionalan u train i val skupu
<br>
Split je urađen na nivou cele slike, pre cropovanja na 512×512 (na crop-ovima se radi trening), čime je sprečeno curenje podataka (crops iz iste slike ne mogu završiti u
različitim skupovima).<br>
Rezultujuće liste image-ID-jeva su zamrznute u `data/splits/{train,val,test}_image_ids.txt` i koriste ih svi modeli i evaluacija.